In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV


In [29]:
train = pd.read_csv('dataset/train_cleaned.csv')
test = pd.read_csv('dataset/test_cleaned.csv')

train.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_U,Title,Age_scaled,Fare_scaled
0,1,0,3,0,22.0,1,0,7.2500,False,True,False,False,False,False,False,False,False,True,1,-0.565736,-0.879741
1,2,1,1,1,38.0,1,0,71.2833,False,False,False,True,False,False,False,False,False,False,3,0.663861,1.361220
2,3,1,3,1,26.0,0,0,7.9250,False,True,False,False,False,False,False,False,False,True,2,-0.258337,-0.798540
3,4,1,1,1,35.0,1,0,53.1000,False,True,False,True,False,False,False,False,False,False,3,0.433312,1.062038
4,5,0,3,0,35.0,0,0,8.0500,False,True,False,False,False,False,False,False,False,True,1,0.433312,-0.784179


In [5]:
X = train.drop(columns=['PassengerId', 'Survived','Age','Fare'])
Y = train['Survived']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42,stratify=Y)

X_t , X_cv , Y_t , Y_cv = train_test_split(X_test, Y_test, test_size=0.2, random_state=42,stratify=Y_test)


In [ ]:
# Logistic Regression
param_grid_logistic = {
    'C' : [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty' : ['l1', 'l2'],
    'solver' : ['liblinear','lbfgs','saga'],
    'max_iter' : [100, 1000, 10000]
}

model_logistic = GridSearchCV(LogisticRegression(), param_grid_logistic, scoring='accuracy',cv=5, verbose=1, n_jobs=-1)

model_logistic.fit(X_train,Y_train)

print(f'Best parameters: {model_logistic.best_params_}')
print("Best CV score:", model_logistic.best_score_)



Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'C': 0.1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV score: 0.8201677419354839


c:\Users\longo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
90 fits failed out of a total of 540.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
90 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\longo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\longo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\longo\AppDa

In [14]:
logistic_pred = model_logistic.predict(X_cv)
print(f"Accuracy Score for Logistic Regression: {accuracy_score(Y_cv,logistic_pred)}")


Accuracy Score for Logistic Regression: 0.7777777777777778


In [15]:
# Decision Tree


from sklearn.tree import DecisionTreeClassifier

param_grid_tree = {
    'max_depth' : [1,2,3,4,5,6,7,8,9,10,None],
    'min_samples_split' : [2,3,4,5,6,7,8,9,10,15,20],
    'min_samples_leaf' : [1,2,3,4,5,6,7,8,9,10],
    'criterion' : ['gini', 'entropy']
}

model_tree = GridSearchCV(DecisionTreeClassifier(), param_grid_tree, scoring='accuracy',cv=5, verbose=1, n_jobs=-1)

model_tree.fit(X_train,Y_train)

print(f'Best parameters: {model_tree.best_params_}')
print("Best CV score:", model_tree.best_score_)


Fitting 5 folds for each of 2420 candidates, totalling 12100 fits
Best parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 6}
Best CV score: 0.8330322580645161


In [16]:
tree_pred = model_tree.predict(X_cv)
print(f"Accuracy Score for Decision Tree: {accuracy_score(Y_cv,tree_pred)}")

Accuracy Score for Decision Tree: 0.7592592592592593


In [17]:
# XGBoost

from xgboost import XGBClassifier

param_grid_xgb ={
    'n_estimators': [100, 200, 500],
    'max_depth': [3, 4, 5,10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

model_xgb = GridSearchCV(XGBClassifier(), param_grid_xgb, scoring='accuracy',cv=5, verbose=1, n_jobs=-1)

model_xgb.fit(X_train,Y_train)

print(f'Best parameters: {model_xgb.best_params_}')
print("Best CV score:", model_xgb.best_score_)


Fitting 5 folds for each of 144 candidates, totalling 720 fits
Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200, 'subsample': 1.0}
Best CV score: 0.8442322580645161


In [18]:
xgb_pred = model_xgb.predict(X_cv)
print(f"Accuracy Score for XGBoost: {accuracy_score(Y_cv,xgb_pred)}")

Accuracy Score for XGBoost: 0.7592592592592593


In [ ]:
logistic_final_test_pred = model_logistic.predict(X_t)
print(f"Accuracy Score for Logistic Regression: {accuracy_score(Y_t,logistic_final_pred)}")


Accuracy Score for Logistic Regression: 0.8317757009345794


In [34]:
train_columns = X_train.columns

X_Test = test.drop(columns=['PassengerId','Age','Fare'])
X_Test = X_Test[train_columns]
Pred_on_test = model_xgb.predict(X_Test)

In [35]:
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': Pred_on_test})

submission.to_csv('submission.csv', index=False)

In [36]:
submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   PassengerId  418 non-null    int64
 1   Survived     418 non-null    int32
dtypes: int32(1), int64(1)
memory usage: 5.0 KB
